In [1]:
from io import StringIO

class InvalidRecord(Exception):
    pass


def parse_line(line):

    parts = line.strip().split(",")

    if len(parts) != 3:
        raise InvalidRecord(
            f"expected 3 fields, found {len(parts)}"
        )

    reg_no, date, status = (
        part.strip() for part in parts
    )

    if status not in ("P", "A"):
        raise InvalidRecord(f"invalid status {status!r}")

    if not reg_no or not date:
        raise InvalidRecord("register number and date cannot be empty")

    return reg_no, date, status


def summarise(data):

    present = 0
    absent = 0
    errors = []

    try:
        with StringIO(data) as file:

            for line_number, line in enumerate(file, start=1):

                if not line.strip():
                    continue

                try:
                    _, _, status = parse_line(line)

                    if status == "P":
                        present += 1
                    else:
                        absent += 1

                except InvalidRecord as error:
                    errors.append((line_number, str(error)))

    except PermissionError as error:
        raise PermissionError("Cannot read attendance data") from error

    else:
        return present, absent, errors

    finally:
        print("Attendance processing completed.")


if __name__ == "__main__":

    sample_data = """192311001,2026-10-08,P
192311002,2026-10-08,A
192311003,2026-10-08
192311004,2026-10-08,L
"""

    print(summarise(sample_data))

Attendance processing completed.
(1, 1, [(3, 'expected 3 fields, found 2'), (4, "invalid status 'L'")])
